<a href="https://colab.research.google.com/github/hania-sajjad/WEEK-6-TASK/blob/main/notebooks/week6_advanced_nlp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 6: Advanced Natural Language Processing (NLP)

## BBC News Analysis using Named Entity Recognition, Topic Modeling, and Transfer Learning


---

## Project Overview

Natural Language Processing (NLP) enables computers to understand, analyze, and derive meaningful insights from human language. In this project, three advanced NLP techniques are applied to the BBC News dataset to explore different aspects of text analysis.

The project consists of three major components:

- **Named Entity Recognition (NER):** Extract and analyze entities such as people, organizations, and locations from news articles.
- **Topic Modeling:** Discover hidden themes across the collection of articles using an unsupervised learning approach.
- **Transfer Learning:** Classify news articles into predefined categories using a pre-trained transformer model and compare its performance with a traditional machine learning baseline.

These techniques represent a complete NLP pipeline commonly used in applications such as news aggregation, media analytics, business intelligence, and information retrieval.

In [19]:
# Import Libraries

# Data Manipulation
import pandas as pd
import numpy as np

# Data Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Text Processing
import re
import string
import nltk

# spaCy for Named Entity Recognition
import spacy
from collections import Counter

# Topic Modeling
#!pip install gensim #not available by default
import gensim
from gensim import corpora
from gensim.models import CoherenceModel

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    precision_score,
    recall_score,
    f1_score
)

# Transformers
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments
)

from datasets import Dataset

# PyTorch
import torch

# Ignore warnings
import warnings
warnings.filterwarnings("ignore")

# Set visualization style
sns.set_style("whitegrid")

print("Libraries imported successfully!")

Libraries imported successfully!


# Download Required NLP Resources

Several NLP libraries require additional language resources before they can be used.

In this section, the required datasets, tokenizers, stopwords, and language models are downloaded. These resources will support text preprocessing, Named Entity Recognition, and topic modeling throughout the project.

In [20]:
# Download NLTK resources
nltk.download("punkt")
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

# Download spaCy English model
#!python -m spacy download en_core_web_sm

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

# Load the BBC News Dataset

The BBC News dataset is loaded into a Pandas DataFrame for analysis.

After loading the dataset, we will inspect its structure, verify that it has been read correctly, and ensure that it is suitable for further preprocessing and analysis.

In [21]:
from google.colab import drive
drive.mount('/content/drive')

df = pd.read_csv("/content/drive/MyDrive/Data/week6.csv")

print("Dataset loaded successfully!\n")

print(f"Dataset Shape: ")
print(df.shape)

df.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset loaded successfully!

Dataset Shape: 
(2127, 7)


,text,labels,no_sentences,Flesch Reading Ease Score,Dale-Chall Readability Score,text_rank_summary,lsa_summary
0,Ad sales boost Time Warner profit\n\nQuarterly...,business,26,62.17,9.72,It hopes to increase subscribers by offering t...,Its profits were buoyed by one-off gains which...
1,Dollar gains on Greenspan speech\n\nThe dollar...,business,17,65.56,9.09,The dollar has hit its highest level against t...,"""I think the chairman's taking a much more san..."
2,Yukos unit buyer faces loan claim\n\nThe owner...,business,14,69.21,9.66,The owners of embattled Russian oil giant Yuko...,Yukos' owner Menatep Group says it will ask Ro...
3,High fuel prices hit BA's profits\n\nBritish A...,business,24,62.98,9.86,Looking ahead to its full year results to Marc...,"Rod Eddington, BA's chief executive, said the ..."
4,Pernod takeover talk lifts Domecq\n\nShares in...,business,17,70.63,10.23,Reports in the Wall Street Journal and the Fin...,Shares in UK drinks and food firm Allied Domec...


In [22]:
df.info()
df.columns


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2127 entries, 0 to 2126
Data columns (total 7 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   text                          2127 non-null   object 
 1   labels                        2127 non-null   object 
 2   no_sentences                  2127 non-null   int64  
 3   Flesch Reading Ease Score     2127 non-null   float64
 4   Dale-Chall Readability Score  2127 non-null   float64
 5   text_rank_summary             2127 non-null   object 
 6   lsa_summary                   2127 non-null   object 
dtypes: float64(2), int64(1), object(4)
memory usage: 116.4+ KB


Index(['text', 'labels', 'no_sentences', 'Flesch Reading Ease Score',
       'Dale-Chall Readability Score', 'text_rank_summary', 'lsa_summary'],
      dtype='object')

In [23]:
# Rename labels to "category"
df.rename(columns={"labels": "category"}, inplace=True)